# 00. Sincronización de Metadatos y Descarga de Datos Crudos

Este cuaderno automatiza dos tareas esenciales previas al preprocesamiento:
1. **Sincronización de Metadatos**: Lee la configuración de `metadata/registry.json`, consulta la API CKAN de Datos Abiertos Perú (`datosabiertos.gob.pe`) y genera/actualiza el catálogo unificado en `metadata/datasets.json`. Si en el futuro se agregan más meses u otras empresas en `registry.json`, este cuaderno actualiza automáticamente los enlaces y formatos.
2. **Descarga de Datos Crudos**: Descarga los archivos mensuales oficiales (CSV, XLSX o ZIP) directamente a `data/raw/{dataset}/`, saltando aquellos que ya existen para no saturar la red ni desperdiciar tiempo.


In [ ]:
from __future__ import annotations

import json
import re
import time
import urllib.request
from datetime import datetime
from pathlib import Path
import pandas as pd
from IPython.display import display

# Rutas del proyecto
ROOT = Path.cwd().resolve()
METADATA_DIR = ROOT / "metadata"
REGISTRY_PATH = METADATA_DIR / "registry.json"
DATASETS_JSON_PATH = METADATA_DIR / "datasets.json"
RAW_BASE_DIR = ROOT / "data" / "raw"

METADATA_DIR.mkdir(parents=True, exist_ok=True)
RAW_BASE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raíz:          {ROOT}")
print(f"Registro:      {REGISTRY_PATH}")
print(f"Catálogo JSON: {DATASETS_JSON_PATH}")
print(f"Carpeta RAW:   {RAW_BASE_DIR}")


## 1. Funciones de Sincronización con la API CKAN

Mapea los recursos de datos abiertos a su período mensual canónico (`YYYYMM`) y asigna los lectores correspondientes (incluyendo los *overrides* de `registry.json`).


In [ ]:
MONTH_MAP = {
    "ene": "01", "enero": "01",
    "feb": "02", "febrero": "02",
    "mar": "03", "marzo": "03",
    "abr": "04", "abril": "04",
    "may": "05", "mayo": "05",
    "jun": "06", "junio": "06",
    "jul": "07", "julio": "07",
    "ago": "08", "agost": "08", "agosto": "08",
    "set": "09", "sep": "09", "setiembre": "09", "septiembre": "09",
    "oct": "10", "octubre": "10",
    "nov": "11", "noviembre": "11",
    "dic": "12", "diciembre": "12",
}

CKAN_API_BASE = "https://www.datosabiertos.gob.pe/api/3/action/package_show"


def extract_period(resource_name: str, resource_url: str) -> str | None:
    """Extrae el período YYYYMM desde el nombre del recurso o su URL."""
    name_clean = resource_name.lower().replace(".", " ").replace("-", " ")
    match = re.search(
        r"\b(ene(?:ro)?|feb(?:rero)?|mar(?:zo)?|abr(?:il)?|may(?:o)?|jun(?:io)?|jul(?:io)?|ago(?:st|sto)?|set(?:iembre)?|sep(?:tiembre)?|oct(?:ubre)?|nov(?:iembre)?|dic(?:iembre)?)\s*(\d{4})\b",
        name_clean
    )
    if match:
        month_str, year = match.groups()
        month_num = MONTH_MAP.get(month_str[:3])
        if month_num:
            return f"{year}{month_num}"

    url_clean = resource_url.lower().replace(".", "_").replace("-", "_")
    url_month_match = re.search(
        r"_(ene(?:ro)?|feb(?:rero)?|mar(?:zo)?|abr(?:il)?|may(?:o)?|jun(?:io)?|jul(?:io)?|ago(?:st|sto)?|set(?:iembre)?|sep(?:tiembre)?|oct(?:ubre)?|nov(?:iembre)?|dic(?:iembre)?)_?(\d{4})",
        url_clean
    )
    if url_month_match:
        month_str, year = url_month_match.groups()
        month_num = MONTH_MAP.get(month_str[:3])
        if month_num:
            return f"{year}{month_num}"

    url_filename = resource_url.split("/")[-1]
    url_match = re.search(r"20\d{4}", url_filename)
    if url_match:
        return url_match.group(0)

    ddmmyy_match = re.search(r"_(\d{2})(\d{2})(\d{2})\.csv", url_filename)
    if ddmmyy_match:
        _, mm, yy = ddmmyy_match.groups()
        return f"20{yy}{mm}"

    return None


def fetch_package_resources(package_id: str) -> list[dict]:
    """Consulta los recursos disponibles de un paquete en la API de Datos Abiertos."""
    url = f"{CKAN_API_BASE}?id={package_id}"
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    )
    with urllib.request.urlopen(req) as resp:
        data = json.loads(resp.read().decode("utf-8"))
        result = data.get("result", [])
        pkg = result[0] if isinstance(result, list) and result else result
        return pkg.get("resources", [])


def determine_default_reader(filename: str, format_str: str) -> str:
    ext = Path(filename).suffix.lower()
    if ext == ".xlsx" or format_str.lower() in {".xlsx", "xlsx"}:
        return "xlsx"
    return "csv"


def sync_metadata(force_online: bool = True) -> dict:
    """Sincroniza registry.json con la API CKAN y actualiza metadata/datasets.json."""
    if not REGISTRY_PATH.exists():
        raise FileNotFoundError(f"No existe {REGISTRY_PATH}")
    
    registry = json.loads(REGISTRY_PATH.read_text(encoding="utf-8"))
    
    catalog = {
        "_comment": "AUTO-GENERATED FILE. Actualizado desde 00_descarga.ipynb",
        "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "version": "1.0",
        "datasets": {}
    }
    
    for dataset_key, meta in registry.items():
        print(f"[*] Consultando API CKAN para: {meta['name']} ({dataset_key})...")
        resources = fetch_package_resources(meta["package_id"]) if force_online else []
        
        sources = {}
        seen_urls = set()
        
        for r in resources:
            res_url = r.get("url", "").strip()
            res_name = r.get("name", "").strip()
            res_format = r.get("format", "").strip()
            
            if not res_url or res_url in seen_urls:
                continue
            if "diccionario" in res_name.lower() or "diccionario" in res_url.lower():
                continue
                
            period = extract_period(res_name, res_url)
            if not period or period in sources:
                continue
                
            seen_urls.add(res_url)
            filename = urllib.request.unquote(res_url.split("/")[-1])
            
            overrides = meta.get("overrides", {}).get(period)
            if overrides:
                ingestion = dict(overrides)
            else:
                ingestion = {"reader": determine_default_reader(filename, res_format)}
                
            if "columns_schema" in meta and "columns_schema" not in ingestion:
                ingestion["columns_schema"] = meta["columns_schema"]
                
            sources[period] = {
                "source": {"url": res_url, "filename": filename},
                "ingestion": ingestion
            }
            
        sorted_sources = {k: sources[k] for k in sorted(sources.keys())}
        catalog["datasets"][dataset_key] = {
            "name": meta["name"],
            "provider": meta["provider"],
            "domain": meta["domain"],
            "schema": meta["schema"],
            "columns_schema": meta.get("columns_schema", "standard"),
            "frequency": meta["frequency"],
            "storage": meta["storage"],
            "sources": sorted_sources
        }
        print(f"    -> {len(sorted_sources)} períodos sincronizados con éxito.")
        
    DATASETS_JSON_PATH.write_text(json.dumps(catalog, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"\n[OK] Catálogo guardado en: {DATASETS_JSON_PATH}")
    return catalog


## 2. Ejecutar Sincronización de Metadatos


In [ ]:
catalog = sync_metadata(force_online=True)

# Vista previa de lo sincronizado
resumen_sync = []
for k, v in catalog["datasets"].items():
    resumen_sync.append({
        "dataset_key": k,
        "empresa": v["provider"],
        "esquema": v["columns_schema"],
        "total_periodos": len(v["sources"]),
        "primer_periodo": min(v["sources"].keys()) if v["sources"] else None,
        "ultimo_periodo": max(v["sources"].keys()) if v["sources"] else None,
    })

display(pd.DataFrame(resumen_sync))


## 3. Funciones de Descarga de Datos Crudos

Descarga segura con escritura atómica (`.tmp`) y verificación de archivos ya existentes para evitar re-descargas.


In [ ]:
CHUNK_SIZE = 1024 * 1024  # 1MB


def download_file(url: str, dest_path: Path) -> bool:
    """Descarga un archivo con progreso atómico."""
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = dest_path.with_suffix(dest_path.suffix + ".tmp")
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
    req = urllib.request.Request(url, headers=headers)
    
    try:
        with urllib.request.urlopen(req) as resp:
            total_size = int(resp.headers.get("Content-Length", 0))
            downloaded = 0
            with temp_path.open("wb") as out_file:
                while True:
                    chunk = resp.read(CHUNK_SIZE)
                    if not chunk:
                        break
                    out_file.write(chunk)
                    downloaded += len(chunk)
                    if total_size > 0:
                        pct = (downloaded / total_size) * 100
                        print(f"\r    -> Descargando: {downloaded / (1024*1024):.1f}MB / {total_size / (1024*1024):.1f}MB ({pct:.1f}%)", end="", flush=True)
            print()
        temp_path.replace(dest_path)
        return True
    except Exception as e:
        if temp_path.exists():
            temp_path.unlink()
        print(f"\n    [Error al descargar {url}]: {e}")
        return False


def download_dataset_raw(dataset_key: str, force: bool = False):
    """Descarga los archivos faltantes de un dataset hacia data/raw/{dataset}."""
    spec = catalog["datasets"].get(dataset_key)
    if not spec:
        print(f"Dataset {dataset_key} no encontrado en el catálogo.")
        return
        
    raw_dir = ROOT / spec.get("storage", {}).get("raw", f"data/raw/{dataset_key}")
    raw_dir.mkdir(parents=True, exist_ok=True)
    sources = spec.get("sources", {})
    
    print(f"\n==========================================")
    print(f"Descargando datos crudos para: {spec['name']}")
    print(f"Destino: {raw_dir}")
    print(f"==========================================")
    
    downloaded = 0
    skipped = 0
    
    for period, item in sources.items():
        url = item["source"]["url"]
        filename = item["source"]["filename"]
        dest_file = raw_dir / filename
        
        # Verificar si ya existe con esa u otra extensión equivalente (.zip, .csv, .xlsx)
        stem = Path(filename).stem
        exists = dest_file.exists() and dest_file.stat().st_size > 0
        if not exists:
            for ext in [".zip", ".csv", ".xlsx"]:
                cand = raw_dir / f"{stem}{ext}"
                if cand.exists() and cand.stat().st_size > 0:
                    exists = True
                    dest_file = cand
                    break
                    
        if exists and not force:
            skipped += 1
            continue
            
        print(f"[{period}] Descargando: {filename}...")
        ok = download_file(url, dest_file)
        if ok:
            downloaded += 1
            
    print(f"Resumen {dataset_key}: {downloaded} descargados nuevos, {skipped} ya existentes.")


## 4. Ejecución de la Descarga

Descargamos los archivos para **ENOSA** (y opcionalmente para otros datasets si están configurados en `registry.json`).


In [ ]:
# Descargar archivos de ENOSA (salta automáticamente los que ya existen)
download_dataset_raw("enosa_energy_consumption", force=False)


## 5. Verificación de Archivos Crudos en Disco


In [ ]:
raw_enosa = ROOT / "data" / "raw" / "enosa"
archivos_enosa = sorted(raw_enosa.glob("*.*"))

print(f"Total archivos en {raw_enosa}: {len(archivos_enosa)}\n")
lista_archivos = []
for f in archivos_enosa:
    lista_archivos.append({
        "archivo": f.name,
        "extension": f.suffix,
        "tamano_mb": round(f.stat().st_size / (1024 * 1024), 2),
    })

df_raw = pd.DataFrame(lista_archivos)
display(df_raw.head(10))
print(f"Peso total en disco: {df_raw['tamano_mb'].sum():.2f} MB")
